In [1]:
import os
import json
import pandas as pd

In [2]:
mis_adm_sqlite = "../../backend/source/administrator.sqlite"
flow_adms_path = "./output/flow_sqlite/"
flow_adms = pd.DataFrame(columns=[    
    "flow_form_id",
    "flow_question_id",
    "flow_value",
    "mis_value",
])
flow_missing_adms = pd.DataFrame(columns=[
    "id",
    "code",
    "name",
    "full_path",
])

In [3]:
flow_cascades_path = "./output/flow_cascades"
all_dfs = []
# loop through all csv files in flow_cascades_path
for fname in filter(lambda f: f.endswith('.csv'), os.listdir(flow_cascades_path)):
    file_path = os.path.join(flow_cascades_path, fname)
    # read csv file using pandas
    df = pd.read_csv(file_path)
    all_dfs.append(df)

# combine into one DataFrame
flow_cascades_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

# group by cascade_resource and print each group
if "cascade_resource" not in flow_cascades_df.columns:
    print("Column 'cascade_resource' not found. Available columns:", list(flow_cascades_df.columns))
else:
    survey_grouped = flow_cascades_df.groupby("survey_id")
    missing_adms = {}
    for s_id, s_rows in survey_grouped:
        # Load answers from csv file in ./output/flow_data/
        # find csv file in ./output/flow_data/ that prefix matches survey id
        data_files = list(filter(lambda f: f.startswith(str(s_id)) and f.endswith('.csv'), os.listdir("./output/flow_data/")))
        if not data_files:
            print(f"No data file found for survey_id {s_id}")
            continue
        data_file_path = os.path.join("./output/flow_data/", data_files[0])
        data_df = pd.read_csv(data_file_path)
        for _, row in s_rows.iterrows():
            question_id = row['question_id']
            sqlite_file = row['cascade_resource']
            # check if question_id column exists in data_df
            if str(question_id) not in data_df.columns:
                print(f"    Question ID {question_id} not found in data for survey {s_id}")
                continue
            # all answers in question_id column
            answers = data_df[str(question_id)]
            missing_adms.setdefault(sqlite_file, [])
            for answer_value in answers.dropna():
                # JSON parse answer_value if possible
                flow_value = answer_value
                json_parsed = None
                mis_value = None
                try:
                    json_parsed = json.loads(answer_value)
                    flow_value = "|".join([
                        str(v["name"]).strip()
                        for v in json_parsed
                    ])
                    # get last value in json_parsed as adm_value
                    adm_value = json_parsed[-1]["name"]
                    if adm_value:
                        adm_value = adm_value.lower().strip()
                    adm_level = len(json_parsed) + 1 # +1 Add national level
                    adm_query = "SELECT * FROM nodes WHERE LOWER(TRIM(name)) = ? AND level = ?;"
                    adm_match = pd.read_sql_query(
                        adm_query,
                        f"sqlite:///{mis_adm_sqlite}",
                        params=(adm_value, adm_level),
                    )
                    if not adm_match.empty:
                        mis_value = adm_match.iloc[0]['id']
                except (json.JSONDecodeError, TypeError):
                    pass
                if mis_value is None:
                    if flow_value not in missing_adms[sqlite_file]:
                        missing_adms[sqlite_file].append(flow_value)
                flow_adms = pd.concat([
                    flow_adms,
                    pd.DataFrame([{
                        "flow_form_id": s_id,
                        "flow_question_id": question_id,
                        "flow_value": flow_value,
                        "mis_value": mis_value,
                    }])
                ], ignore_index=True)
    # process missing_adms into flow_missing_adms DataFrame
    for sqlite_file, adm_list in missing_adms.items():
        flow_sqlite = os.path.join(flow_adms_path, sqlite_file)
        for adm in adm_list:
            adm_value = adm.split("|")[-1].lower().strip()
            adm_parent = adm.split("|")[-2] if len(adm.split("|")) > 1 else None
            # create self join query to find name and parent.name in nodes table
            adm_query = """
            SELECT n1.* FROM nodes n1
            LEFT JOIN nodes n2 ON n1.parent = n2.id
            WHERE LOWER(TRIM(n1.name)) = ? AND n2.name = ?;
            """
            adm_match = pd.read_sql_query(
                adm_query,
                f"sqlite:///{flow_sqlite}",
                params=(adm_value, adm_parent),
            )
            if adm_match.empty:
                continue
            flow_missing_adms = pd.concat([
                flow_missing_adms,
                pd.DataFrame([{
                    "id": adm_match.iloc[0]['id'],
                    "code": adm_match.iloc[0]['code'],
                    "name": adm_match.iloc[0]['name'],
                    "full_path": adm,
                }])
            ], ignore_index=True)
# save flow_adms to csv
flow_adms.to_csv("../../backend/source/akvo-flow/administration_mapping.csv", index=False)
print("Flow administration mapping saved to ../../backend/source/akvo-flow/administration_mapping.csv")
# save flow_missing_adms to csv
flow_missing_adms.to_csv("../../backend/source/akvo-flow/administration_missing.csv", index=False)
print("Flow missing administration saved to ../../backend/source/akvo-flow/administration_missing.csv")

Flow administration mapping saved to ../../backend/source/akvo-flow/administration_mapping.csv
Flow missing administration saved to ../../backend/source/akvo-flow/administration_missing.csv


In [4]:
# if mis_adm_sqlite does not exist, raise error
if not os.path.exists(mis_adm_sqlite):
    raise FileNotFoundError(f"{mis_adm_sqlite} not found. Please run `python manage.py generate_sqlite` in the backend container to create the database.")

In [5]:
flow_cascades_path = "./output/flow_cascades"
all_dfs = []
# loop through all csv files in flow_cascades_path
for fname in filter(lambda f: f.endswith('.csv'), os.listdir(flow_cascades_path)):
    file_path = os.path.join(flow_cascades_path, fname)
    # read csv file using pandas
    df = pd.read_csv(file_path)
    all_dfs.append(df)

# combine into one DataFrame
flow_cascades_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

# group by cascade_resource and print each group
if "cascade_resource" not in flow_cascades_df.columns:
    print("Column 'cascade_resource' not found. Available columns:", list(flow_cascades_df.columns))
else:
    # Pre-load all MIS administration data to avoid N+1 queries
    print("Pre-loading MIS administration data...")
    mis_adm_data = pd.read_sql_query(
        "SELECT id, LOWER(TRIM(name)) as name_lower, level FROM nodes",
        f"sqlite:///{mis_adm_sqlite}"
    )
    # Create a lookup dictionary for faster matching: (name_lower, level) -> id
    mis_adm_lookup = {
        (row['name_lower'], row['level']): row['id'] 
        for _, row in mis_adm_data.iterrows()
    }
    print(f"Loaded {len(mis_adm_lookup)} MIS administration entries")
    
    survey_grouped = flow_cascades_df.groupby("survey_id")
    missing_adms = {}
    # Use a map to deduplicate entries early while accumulating answers.
    # Key: (flow_datapoint_id, flow_form_id, flow_question_id)
    flow_adms_map = {}
    for s_id, s_rows in survey_grouped:
        # Load answers from csv file in ./output/flow_data/
        # find csv file in ./output/flow_data/ that prefix matches survey id
        data_files = list(filter(lambda f: f.startswith(str(s_id)) and f.endswith('.csv'), os.listdir("./output/flow_data/")))
        if not data_files:
            print(f"No data file found for survey_id {s_id}")
            continue
        data_file_path = os.path.join("./output/flow_data/", data_files[0])
        data_df = pd.read_csv(data_file_path)
        for _, row in s_rows.iterrows():
            question_id = row['question_id']
            sqlite_file = row['cascade_resource']
            # check if question_id column exists in data_df
            if str(question_id) not in data_df.columns:
                print(f"    Question ID {question_id} not found in data for survey {s_id}")
                continue
            # all answers in question_id column
            answers = data_df[str(question_id)]
            missing_adms.setdefault(sqlite_file, [])
            for answer_value in answers.dropna():
                # JSON parse answer_value if possible
                flow_value = answer_value
                json_parsed = None
                mis_value = None
                try:
                    json_parsed = json.loads(answer_value)
                    flow_value = "|".join([
                        str(v["name"]).strip()
                        for v in json_parsed
                    ])
                    # get last value in json_parsed as adm_value
                    adm_value = json_parsed[-1]["name"]
                    if adm_value:
                        adm_value = adm_value.lower().strip()
                    adm_level = len(json_parsed) + 1 # +1 Add national level
                    # Use pre-loaded lookup instead of querying database
                    lookup_key = (adm_value, adm_level)
                    if lookup_key in mis_adm_lookup:
                        mis_value = int(mis_adm_lookup[lookup_key])
                except (json.JSONDecodeError, TypeError):
                    pass
                if mis_value is None:
                    if flow_value not in missing_adms[sqlite_file]:
                        missing_adms[sqlite_file].append(flow_value)
                # Determine datapoint id for this answer
                datapoint_vals = data_df.loc[answers == answer_value, 'datapoint_id'].values
                if len(datapoint_vals) == 0:
                    # skip if no datapoint_id found
                    continue
                flow_datapoint_id = datapoint_vals[0]
                # Build key and merge into map to deduplicate early
                key = (flow_datapoint_id, s_id, question_id)
                entry = flow_adms_map.get(key)
                if entry is None:
                    flow_adms_map[key] = {
                        'flow_datapoint_id': flow_datapoint_id,
                        'flow_form_id': s_id,
                        'flow_question_id': question_id,
                        'flow_values': set([flow_value]) if pd.notna(flow_value) else set(),
                        'mis_value': mis_value,
                    }
                else:
                    # merge flow_value into set
                    if pd.notna(flow_value):
                        entry['flow_values'].add(flow_value)
                    # prefer a non-null mis_value if we don't already have one
                    if entry.get('mis_value') is None and mis_value is not None:
                        entry['mis_value'] = mis_value
    
    # Convert deduplicated map into a list suitable for DataFrame creation
    flow_adms_list = []
    for (_dpid, _form, _q), v in flow_adms_map.items():
        fv = '|'.join(sorted(v['flow_values'])) if v['flow_values'] else pd.NA
        flow_adms_list.append({
            'flow_datapoint_id': v['flow_datapoint_id'],
            'flow_form_id': v['flow_form_id'],
            'flow_question_id': v['flow_question_id'],
            'flow_value': fv,
            'mis_value': v['mis_value'],
        })
    
    # Create flow_adms DataFrame from accumulated deduplicated list
    flow_adms = pd.DataFrame(flow_adms_list) if flow_adms_list else pd.DataFrame(columns=[
        "flow_datapoint_id",
        "flow_form_id",
        "flow_question_id",
        "flow_value",
        "mis_value",
    ])
    
    # Convert mis_value to Int64 (nullable integer type) if possible
    if not flow_adms.empty:
        try:
            flow_adms['mis_value'] = flow_adms['mis_value'].astype('Int64')
        except Exception:
            pass

    # process missing_adms into flow_missing_adms DataFrame
    flow_missing_adms_list = []
    for sqlite_file, adm_list in missing_adms.items():
        flow_sqlite = os.path.join(flow_adms_path, sqlite_file)
        # Pre-load all flow administration data for this sqlite file
        print(f"Pre-loading flow administration data from {sqlite_file}...")
        flow_adm_data = pd.read_sql_query(
            """
            SELECT n1.id, n1.code, n1.name, n2.name as parent_name,
                   LOWER(TRIM(n1.name)) as name_lower
            FROM nodes n1
            LEFT JOIN nodes n2 ON n1.parent = n2.id
            """,
            f"sqlite:///{flow_sqlite}"
        )
        # Create lookup: (name_lower, parent_name) -> row data
        flow_adm_lookup = {}
        for _, row in flow_adm_data.iterrows():
            key = (row['name_lower'], row['parent_name'])
            flow_adm_lookup[key] = {
                'id': row['id'],
                'code': row['code'],
                'name': row['name']
            }
        
        for adm in adm_list:
            adm_value = adm.split("|")[-1].lower().strip()
            adm_parent = adm.split("|")[-2] if len(adm.split("|")) > 1 else None
            # Use pre-loaded lookup instead of querying database
            lookup_key = (adm_value, adm_parent)
            if lookup_key in flow_adm_lookup:
                adm_match = flow_adm_lookup[lookup_key]
                flow_missing_adms_list.append({
                    "id": adm_match['id'],
                    "code": adm_match['code'],
                    "name": adm_match['name'],
                    "full_path": adm,
                })
    
    # Create flow_missing_adms DataFrame from accumulated list
    flow_missing_adms = pd.DataFrame(flow_missing_adms_list) if flow_missing_adms_list else pd.DataFrame(columns=[
        "id",
        "code",
        "name",
        "full_path",
    ])
# save flow_adms to csv
flow_adms.to_csv("../../backend/source/akvo-flow/administration_mapping.csv", index=False)
print("Flow administration mapping saved to ../../backend/source/akvo-flow/administration_mapping.csv")
# save flow_missing_adms to csv
flow_missing_adms.to_csv("../../backend/source/akvo-flow/administration_missing.csv", index=False)
print("Flow missing administration saved to ../../backend/source/akvo-flow/administration_missing.csv")


Pre-loading MIS administration data...
Loaded 98 MIS administration entries
Pre-loading flow administration data from cascade-5430921-v6.sqlite...
Flow administration mapping saved to ../../backend/source/akvo-flow/administration_mapping.csv
Flow missing administration saved to ../../backend/source/akvo-flow/administration_missing.csv
Flow missing administration saved to ../../backend/source/akvo-flow/administration_missing.csv
